# RSA model for a marked hope-wh utterance

This notebook is a small exploratory model. It asks how listener assumptions about speaker background change the predicted speaker choice for a marked sentence such as `I hope what will happen`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.append(str(Path("..").resolve()))
from src.hope_wh_scenario import (
    L2_ASSUMPTION,
    NATIVE_ASSUMPTION,
    MESSAGES,
    cost_sweep,
    make_model,
    marked_speaker_probability,
    prior_sweep,
)

## Speaker assumptions

The native-speaker assumption gives the marked `hope_what_S` message a higher cost and lower message prior. The L2-speaker assumption treats the marked message as less surprising. The shared setup lives in `src/hope_wh_scenario.py` so the notebook, tests, and figure script do not drift apart.

In [ ]:
native_model = make_model(NATIVE_ASSUMPTION)
l2_model = make_model(L2_ASSUMPTION)

native_marked = marked_speaker_probability(native_model)
l2_marked = marked_speaker_probability(l2_model)

print(f"Native S1(hope_what_S | desire_positive_S): {native_marked:.3f}")
print(f"L2 S1(hope_what_S | desire_positive_S): {l2_marked:.3f}")

## Speaker-choice comparison

Because the current truth table is deterministic, the clearest difference is in `S1(message | object)`: how likely the modeled speaker is to choose the marked message.

In [ ]:
Path("../figures").mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 4))
labels = ["Native", "L2"]
values = [native_marked, l2_marked]
ax.bar(labels, values, color=["#4c78a8", "#f58518"])
ax.set_ylim(0, 1.0)
ax.set_ylabel("S1(hope_what_S | desire_positive_S)")
ax.set_title("Speaker choice for the marked message")
for index, value in enumerate(values):
    ax.text(index, value + 0.02, f"{value:.2f}", ha="center")
fig.tight_layout()
fig.savefig("../figures/speaker_choice.svg")

## Cost and prior sensitivity

The next two plots keep the model small but make the two hand-set assumptions easier to inspect.

In [ ]:
cost_rows = cost_sweep()
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([row[0] for row in cost_rows], [row[1] for row in cost_rows], marker="o", markersize=3)
ax.set_xlabel("Cost assigned to hope_what_S")
ax.set_ylabel("S1(hope_what_S | desire_positive_S)")
ax.set_title("Cost sensitivity")
ax.set_ylim(0, 1.0)
fig.tight_layout()
fig.savefig("../figures/cost_sweep.svg")

prior_rows = prior_sweep()
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([row[0] for row in prior_rows], [row[1] for row in prior_rows], marker="o", markersize=3)
ax.set_xlabel("Prior assigned to hope_what_S")
ax.set_ylabel("S1(hope_what_S | desire_positive_S)")
ax.set_title("Prior sensitivity")
ax.set_ylim(0, 1.0)
fig.tight_layout()
fig.savefig("../figures/prior_sweep.svg")

## Current interpretation

This is still a toy model. The useful part is not the exact numeric output, but the way the model makes assumptions about speaker background explicit. A next step would be to replace hand-set costs and priors with judgment data or with values motivated by a more careful syntactic/semantic analysis.